Date: 01/14/2026

Point of Contact: Abigayle Hodson, Abigayle_Hodson@lbl.gov

Organization: Lawrence Berkeley National Laboratory

Purpose: The purpose of this notebook is to assign one or more treatment trains, using the methodology developed in [El Abbadi et al. 2025](https://www.nature.com/articles/s44221-025-00485-w), to approximately 35 water resource recovery facilities based on unit process data collected by Argonne National Laboratory.

In [76]:
# Import necessary libraries
import pandas as pd
import numpy as np

#display all columns in dataframe
pd.set_option('display.max_columns', None)

# Establish file path for ease of uploads and exports
path = 'C:/Users/skar/Box/FY25_Water Analysis_FCG share/3. Survey data collection/2025 Project (CURRENT)/'

In [77]:
def treatment_train_werf(tt_upadd_yr, bgyr):
  '''
  Function that assigns wastewater treatment facilities with enough unit process information one or more treatment trains
    Parameters:
      tt_upadd_yr = reported unit processes for each facility in pivot table form
      bgyr = year which biogas systems for electricity generation are confirmed to be active
    Returns:
      tt_werf_yr = dataframe with treatment train assignments for each facility that has sufficient unit process information
  '''
  # Create dataframe with treatment train assignments
  tt_werf_yr = tt_upadd_yr

  def assign(name, check, exceptions = []):
    '''
    Core function which assigns a treatment train if a subset of key unit processes has been reported on or before specified year
      Parameters:
        name: treatment train to be assigned, written in Tarallo et al., 2015 naming convention
        check: unit processes that have to be present in order for treatment train to be assigned
        exceptions: unit processes and/or treatment trains that must be absent/not yet assigned in order for treatment train to be assigned
      Returns:
        tt_werf_yr = dataframe with treatment train assignments for each facility that has sufficient unit process information
    '''
    #sum the number of key unit processes present within check for a given treatment train
    #for given treatment train, set value equal to the sum above
    tt_werf_yr[name] = sum(tt_werf_yr[check[i]] for i in range(len(check)))

    #if sum of relevant unit processes is not equal to the number of key unit processes in check, turn off treatment train
    tt_werf_yr.loc[tt_werf_yr[name] != len(check), name] = 0

    #if sum of relevant unit processes is equal to the number of key unit processes in check, turn on treatment train
    tt_werf_yr.loc[tt_werf_yr[name] == len(check), name] = 1

    #if exceptions exist, iterate through exceptions and turn treatment train off if one or more exceptions is met
    if len(exceptions) > 0:
        for exception in exceptions:
            tt_werf_yr.loc[tt_werf_yr[exception] == 1, name] = 0

    return tt_werf_yr

  #assign treatment trains sequentially based on if all key unit processes exist and excepting unit processes/trains do not
  #membrane bioreactor trains
  assign('N1E', ['MBR-BNR','AND',f'BIOGAS_{bgyr}'])
  assign('N1', ['MBR-BNR','AND'],['N1E'])
  assign('N2', ['MBR-BNR','AED'])

  #biological and chemical phosphorus removal trains- priority 1 within activated sludge assignment
  assign('H1E', ['AS_BNR_P','AND','CHEM-P',f'BIOGAS_{bgyr}'])
  assign('H1', ['AS_BNR_P','AND','CHEM-P'],['H1E'])

  #biological phosphorus removal trains- priority 2 in activated sludge assignment
  #G train not assigned if H train has already been assigned
  assign('G6', ['AS_BNR_P','FBI'])
  assign('G5', ['AS_BNR_P','MHI'])
  assign('G3', ['AS_BNR_P','LIME'])
  assign('G2', ['AS_BNR_P','AED'])
  assign('G1E', ['AS_BNR_P','AND',f'BIOGAS_{bgyr}'],['H1E','H1'])
  assign('G1', ['AS_BNR_P','AND'], ['G1E','H1E','H1'])

  #biological nitrogen removal trains- priority 3 in assignment
  #I train not assigned if H or G train has already been assigned
  assign('I6', ['AS_BNR_N','FBI'],['G6'])
  assign('I5', ['AS_BNR_N','MHI'],['G5'])
  assign('I3', ['AS_BNR_N','LIME'],['G3'])
  assign('I2', ['AS_BNR_N','AED'],['G2'])
  assign('I1E', ['AS_BNR_N','AND',f'BIOGAS_{bgyr}'],['H1','H1E','G1','G1E'])
  assign('I1', ['AS_BNR_N','AND'], ['I1E','H1','H1E','G1','G1E'])

  #nitrification trains- priority 3 in assignment
  #F/E trains not assigned if H, G, or I train has already been assigned
  assign('F1E', ['BASIC_AS','AND','NIT',f'BIOGAS_{bgyr}'], ['AS_BNR_N','G1','G1E','H1','H1E','I1','I1E'])
  assign('F1', ['BASIC_AS','AND','NIT'], ['AS_BNR_N','F1E','G1','G1E','H1','H1E','I1','I1E'])
  assign('E2P', ['BASIC_AS','AED','NIT','PRIMARY'], ['AS_BNR_N','G2','I2'])
  assign('E2', ['BASIC_AS','AED','NIT'], ['AS_BNR_N','G2','I2','E2P'])

  #pure oxygen activated sludge trains- priority 4 in activated sludge assignment
  #O train not assigned if E, F, H, G, or I train has already been assigned
  assign('O5', ['AS-PUREO','MHI'],['G5','I5'])
  assign('O6', ['AS-PUREO','FBI'],['G6','I6'])
  assign('O3', ['AS-PUREO','LIME'],['G3','I3'])
  assign('O2', ['AS-PUREO','AED'],['G2','I2','E2','E2P'])
  assign('O1E', ['AS-PUREO','AND',f'BIOGAS_{bgyr}'],['F1','F1E','G1E','G1','I1','I1E','H1','H1E'])
  assign('O1', ['AS-PUREO','AND'], ['F1','F1E','O1E','G1E','G1','I1','I1E','H1','H1E'])

  #trickling filter trains
  #D train assignment can exist in multiple treatment trains alongside activated sludge systems
  assign('D5', ['TF_ALL','MHI'])
  assign('D6', ['TF_ALL','FBI'])
  assign('D3', ['TF_ALL','LIME'])
  assign('D2', ['TF_ALL','AED'])
  assign('D1E', ['TF_ALL','AND',f'BIOGAS_{bgyr}'])
  assign('D1', ['TF_ALL','AND'],['D1E'])

  #basic activated sludge, primary trains- priority 5 in activated sludge assignment
  #B train not assigned if O, E, F, H, G, or I train has already been assigned
  assign('B6', ['BASIC_AS','FBI','PRIMARY'], ['G6','I6','O6'])
  assign('B5', ['BASIC_AS','MHI','PRIMARY'], ['AS-PUREO','G5','I5','O5'])
  assign('B4', ['BASIC_AS','AND','BIODRY','PRIMARY'])
  assign('B3', ['BASIC_AS','LIME','PRIMARY'], ['G3','I3','O3'])
  assign('B2', ['BASIC_AS','AED','PRIMARY'], ['E2','E2P','G2','I2','N2','O2'])
  assign('B1E', ['BASIC_AS','AND','PRIMARY',f'BIOGAS_{bgyr}'],['AS_BNR_N','AS-PUREO','B4','F1','F1E','G1','G1E','H1','H1E','I1','I1E','N1','N1E','O1','O1E'])
  assign('B1', ['BASIC_AS','AND','PRIMARY'], ['AS_BNR_N','AS-PUREO','B1E','B4','F1','F1E','G1','G1E','H1','H1E','I1','I1E','N1','N1E','O1','O1E'])

  #basic activated sludge trains- priority 6 in activated sludge assignment
  #B train not assigned if B, O, E, F, H, G, or I train has already been assigned
  assign('C5', ['BASIC_AS','MHI'], ['B5','G5','I5','O5'])
  assign('C6', ['BASIC_AS','FBI'], ['B6','G6','I6','O6'])
  assign('C3', ['BASIC_AS','LIME'], ['B3','G3','I3','O3'])
  assign('C2', ['BASIC_AS','AED'], ['B2','E2','E2P','G2','I2','N2','O2'])
  assign('C1E', ['BASIC_AS','AND',f'BIOGAS_{bgyr}'], ['B1','B1E','B4','F1','F1E','G1','G1E','H1','H1E','I1E','I1','N1E','N1','O1','O1E'])
  assign('C1', ['BASIC_AS','AND'], ['B1','B1E','B4','C1E','F1','F1E','G1','G1E','H1','H1E','I1','I1E','N1','N1E','O1','O1E'])

  #identify the number of treatment trains assigned for each facility in the first round of assignment
  tt_werf_yr['TT_IDENTIFIED'] = sum(tt_werf_yr[i] for i in ('I1E','G6','I6','O5','O6','O3','O1E','G5','I5','C5','C6','O2','O1','N1','N1E','N2','I3','I2','I1','H1','H1E','G3','G2','G1','G1E','F1','F1E','E2','E2P','D5','D6','D1','D1E','D3','D2','C3','C2','C1','C1E','B6','B5','B4','B3','B1E','B1','B2'))

  return tt_werf_yr

In [78]:
#upload unit process pivot table
input_example = pd.read_excel(path + 'Official_Top_50_Plants_Data_Repository.xlsx', sheet_name='TT_ds', header=2)
print (input_example.head())

       CWNS_ID PREDICTED_WRRF_TT_CODE  AED  AND  AS  AS-A2O  AS-BDENIT  AS-EA  \
0  17000721001                   *AG2    0    1   1       0          0      0   
1  26000596001                *B6, *B    0    0   0       0          0      0   
2  17000721009                   *AG2    0    1   1       0          0      0   
3  11000001001                 *AEF2e    0    1   1       0          1      0   
4  29001023002                   *AE5    0    0   1       0          0      0   

   AS-OD  AS-P  AS-PUREO  AS-SA  AS-SBR  BDENIT  BIO-P  BIODRY  BNIT  BNR  \
0      0     1         0      0       0       0      1       1     0    0   
1      0     0         1      0       0       0      0       0     0    0   
2      0     1         0      0       0       0      1       1     0    0   
3      0     0         0      0       0       1      0       1     0    0   
4      0     0         0      0       0       0      0       0     0    0   

   CHEM-P  DISINF-O3  FBI  LAND_TRT  LIME  MBR-BNR

In [79]:
#for wwtps with sufficient unit process data, assign treatment trains
results = treatment_train_werf(input_example, 2022)

#for treatment O1 trains that have nitrification, switch to F1
index = results.loc[(results['O1'] == 1) & (results['NIT'] == 1)].index
for i in index:
  results.at[i,'F1'] = 1
  results.at[i,'O1'] = 0

#for treatment O1E trains that have nitrification, switch to F1E
index2 = results.loc[(results['O1E'] == 1) & (results['NIT'] == 1)].index
for i in index2:
  results.at[i,'F1E'] = 1
  results.at[i,'O1E'] = 0

#convert treatment train assignment naming convention to match El Abbadi et al. 2025
crosswalk = {'B1':'*A1',
             'B1E':'*A1e',
             'B2':'*A3',
             'B3':'*A4',
             'B4':'*A2',
             'B5':'*A5',
             'B6':'*A6',
             'C1':'A1',
             'C1E':'A1e',
             'C2':'A3',
             'C3':'A4',
             'C5':'A5',
             'C6':'A6',
             'D1':'*C1',
             'D1E':'*C1e',
             'D2':'*C3',
             'D3':'*C4',
             'D5':'*C5',
             'D6':'*C6',
             'E2':'E3',
             'E2P':'*E3',
             'F1':'*E1',
             'F1E':'*E1e',
             'G1':'*G1',
             'G1E':'*G1e',
             'G2':'*G3',
             'G3':'*G4',
             'G5':'*G5',
             'G6':'*G6',
             'H1':'*G1-p',
             'H1E':'*G1e-p',
             'I1':'F1',
             'I1E':'F1e',
             'I2':'F3',
             'I3':'F4',
             'I5':'F5',
             'I6':'F6',
             'LAGOON_AER':'L-a',
             'LAGOON_ANAER':'L-n',
             'LAGOON_FAC':'L-f',
             'LAGOON_UNCATEGORIZED':'L-u',
             'N1':'*D1',
             'N1E':'*D1e',
             'N2':'*D3',
             'O1':'*B1',
             'O1E':'*B1e',
             'O2':'*B3',
             'O3':'*B4',
             'O5':'*B5',
             'O6':'*B6'}
results.rename(columns = crosswalk, inplace = True)

#export treatment train assignments to a csv
results.to_csv(path + 'TT_ds_results.csv', index = False)